# Tutorial 51: estimate first, run second — the two-phase values API

The same CNN round-trip as tutorial 50, but with the analysis request and the
execution request as separate calls:

1. **`client.estimate_code(...)`** sends the code block to cas-analyzer and
   returns the `TaskClassification` — the quote (tier, VRAM, CU). Nothing is
   submitted, nothing is charged.
2. **`client.run_code(..., classification=...)`** submits with the precomputed
   classification — the analysis phase is skipped, the code is analyzed once.

This is the flow interactive adapters build on: show the user the quote,
let them decide, then execute.


In [1]:
%env CAS_CLIENT_CONFIG=../../cas-client/.env

env: CAS_CLIENT_CONFIG=../../cas-client/.env


In [2]:
from krauncher import KrauncherClient
from krauncher.values import decode_outputs

client = KrauncherClient()

# The code block runs remotely as-is: `epochs`, `batch_size` and `val_images`
# arrive from inputs=...; `losses`, `val_preds` and `device_name` are picked
# up from its namespace by outputs=[...].
TRAIN_CODE = """
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(self.act(self.bn(self.conv(x))))

class ImageClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 512),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier(num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
losses = []
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    n_batches = 50
    for i in range(n_batches):
        images = torch.randn(batch_size, 3, 64, 64, device=device)
        labels = torch.randint(0, 10, (batch_size,), device=device)

        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()

    avg = epoch_loss / n_batches
    losses.append(round(avg, 4))
    print(f"Epoch {epoch}/{epochs}  avg_loss={avg:.4f}")

# Classify the validation batch shipped from the client's namespace.
model.eval()
with torch.no_grad():
    val = torch.tensor(val_images, dtype=torch.float32, device=device)
    val_preds = model(val).argmax(dim=1).tolist()
device_name = str(device)
"""

In [3]:
import random

# A small validation batch generated locally: 8 images, 3x32x32.
# Plain nested lists — JSON-safe, well inside the 16 MB inline budget.
val_images = [
    [[[random.random() for _ in range(32)] for _ in range(32)] for _ in range(3)]
    for _ in range(8)
]

INPUTS = {"epochs": 3, "batch_size": 32, "val_images": val_images}
OUTPUTS = ["losses", "val_preds", "device_name"]

# Phase 1 — analysis request: classify only, no submission, no charge.
cls = await client.estimate_code(TRAIN_CODE, inputs=INPUTS, outputs=OUTPUTS)
print(f"Quote: tier={cls.tier}, vram={cls.min_vram_gb}GB, "
      f"CU={cls.compute_units}, method={cls.analysis_method}")


  CU: iterations from AST loop analysis=150
  CU: CNN classification (FLOPs): ×0.7 × (img=224/224)^0.65=1.0000 → ×0.7000
  CU: num_calls=3, iters_per_call=50 → total_iters=150
  CU: dps=small (data_gb=1.9), cps=small (tflops=0.031)
  CU: weights(cv_training/small/small/small/few/small): gpu=0.2, stor=0.3, cpu=0.4, pcie=0.1, net=0.01, mem=0.0
  CU: pip_baseline: 1s
  CU: t_io_ref = (0 + 0) / 200 + 1 = 1.00s
  CU: K_algo[A](cv_training, small/small/small, few/small)=41.7147
  CU: t_compute_ref = 1.00 × (0.60/0.41) × 41.7147 = 61.05s
  CU: CU = cu_setup(6000) + cu_io(2000) + cu_warmup(0) + cu_compute(122092) = 130092
Classification: tier=light, VRAM=1GB, CU=130091.8 (compute=122091.8, io=2000.0), method=ast, workload=cv_training, model_size=small, working_set=medium, data/step=small(1.9GB), compute/step=small(0.03TF), profile=[ci=0.30,si=0.10,cu=0.30,pcie=0.40,net=0.05], num_calls=3, iters_per_call=50, num_epochs=3, batch_size=32, time=0.52s


Quote: tier=light, vram=1GB, CU=130091.8, method=ast


In [4]:
# Phase 2 — execution request: submit with the precomputed classification.
# The analysis phase is skipped — the block was classified once, in phase 1.
handle = await client.run_code(
    TRAIN_CODE,
    inputs=INPUTS,
    outputs=OUTPUTS,
    pip=["torch"],
    timeout=300,
    classification=cls,
)
print(f"Task ID: {handle.task_id}")

result = await handle.wait()
values = decode_outputs(result.output, OUTPUTS)

print(f"Losses per epoch: {values['losses']}")
print(f"Validation predictions: {values['val_preds']}")
print(f"Trained on: {values['device_name']}")
print(f"Worker: {result.worker_id}  GPU: {result.actual_gpu}  Time: {result.execution_time_sec:.2f}s")


in-flight atexit sweep installed (IPython kernel detected)


Task ID: 689d4ad7-b131-458b-ae21-9ada26c54bf0
Waiting for worker...
Host obtained: RTX 5060 Ti, 15GB, (vastai)
Waiting for provision (up to 2 min)


[relay] connecting task_id=689d4ad7 target=192.227.212.145:9001 tls=True e2e=True
[relay] TaskStream open task_id=689d4ad7
[relay] msg seq=1 type=event data_keys=['name', 'pub']
[relay] key_exchange received task_id=689d4ad7
[relay] shared key derived task_id=689d4ad7
[relay] uploading payload task_id=689d4ad7 plain_len=502414 enc_len=669924
[relay] payload uploaded ok task_id=689d4ad7
[relay] msg seq=2 type=event data_keys=['name', 'pub']
[event] key_exchange
[relay] msg seq=3 type=event data_keys=['enc']
[event] execution_started


Wait time: 242 sec
Executing on worker-b8261cd8: RTX 5060 Ti, 15GB, (vastai), storage 1852 MB/s, PCIe 13.2 GB/s, net 51 Mbps


[relay] msg seq=4 type=metric data_keys=['enc']
[relay] msg seq=5 type=stdout data_keys=['enc']
[relay] msg seq=6 type=stdout data_keys=['enc']
[relay] msg seq=7 type=stdout data_keys=['enc']
[relay] msg seq=8 type=event data_keys=['enc']
[event] pip_install_complete
[relay] msg seq=9 type=event data_keys=['enc']
[event] execution_complete
[relay] msg seq=10 type=event data_keys=['name']
[relay] stream_ended task_id=689d4ad7


Task 689d4ad7 done in 16.0s — compute 0.0500 KU (€0.0004) — fee 0.8000 KU (€0.0075) — total 0.8500 KU (€0.0079)
Losses per epoch: [2.3412, 2.3108, 2.3041]
Validation predictions: [9, 9, 9, 9, 9, 9, 9, 9]
Trained on: cuda
Worker: worker-b8261cd8  GPU: NVIDIA GeForce RTX 5060 Ti  Time: 15.31s
